# Analysis and Forecasting of BRIT Awards Trends (1982–2025)

**Dataset:** `brit_awards.csv` — 649 rows × 9 columns  
**Source:** Official BRIT Awards Historical Records  
**Coverage:** 100+ award categories × 43 years (1982 – 2025)  
**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn · Scikit-learn  
**ML Task:** Classify award type (`is_person` vs `is_album_or_single`) using Logistic Regression & Random Forest

---

## Section 1 — Data Loading & Initial Exploration
In this section we load the dataset and take a first look at its structure, shape, column names and data types.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('All libraries loaded successfully.')

In [ ]:
# Load dataset
df = pd.read_csv('brit_awards.csv')
print('Dataset loaded successfully.')
print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns')

Each row represents **one award given at one ceremony**. Columns cover the winner name, category, year, venue, host, and boolean flags.

In [ ]:
# First 5 rows
df.head()

In [ ]:
# Last 5 rows
df.tail()

Last records are from **2025** — the most recent ceremony available in the dataset.

In [ ]:
# Column names
print('Columns:', df.columns.tolist())

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Statistical summary
df.describe(include='all')

**Section 1 Summary:**
- 649 rows, 9 columns covering winners from 1982 to 2025
- Mix of object, boolean, and integer columns
- 49 missing values in the `winner` column — to be handled in cleaning

---

## Section 2 — Exploratory Data Analysis (EDA)
Here we explore the dataset to understand distributions, top winners, yearly trends and key statistics.

In [ ]:
# Missing values check
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

In [ ]:
# Duplicate rows check
print(f'Duplicate rows: {df.duplicated().sum()}')

In [ ]:
# Year range and unique categories
print(f'Year Range     : {df["year"].min()} to {df["year"].max()}')
print(f'Unique Categories: {df["details"].nunique()}')
print(f'Unique Venues  : {df["location"].nunique()}')
print(f'Unique Hosts   : {df["host"].nunique()}')
print(f'Unique Winners : {df["winner"].nunique()}')

In [ ]:
# Awards given per year
awards_per_year = df.groupby('year').size().reset_index(name='count')
print('Awards per year (last 10 years):')
print(awards_per_year.tail(10).to_string(index=False))

In [ ]:
# Top 10 most awarded artists (persons only)
top_winners = df[df['is_person'] == True]['winner'].value_counts().head(10)
print('Top 10 Most Awarded Artists:')
print(top_winners.to_string())

In [ ]:
# Top categories by frequency
print('Top 15 Award Categories:')
print(df['details'].value_counts().head(15).to_string())

In [ ]:
# Top venues
print('Top 5 Venues:')
print(df['location'].value_counts().head(5).to_string())

**EDA Summary:**
- 649 records spanning 43 years with 100+ evolving category names
- **U2** leads all-time with 7 wins; Annie Lennox has 6
- **The O2** has hosted the most ceremonies in modern era
- 49 winners are missing — likely historical records with no data
- Award count per year varies between 8 (pandemic year 2020) and 17 (2024)

---

## Section 3 — Data Cleaning
In this section we clean and prepare the raw data — standardizing category names, handling missing values, fixing types and engineering new features.

In [ ]:
# Step 1 — Standardize column names
df.columns = df.columns.str.strip().str.lower()
print('Columns after standardization:', df.columns.tolist())

In [ ]:
# Step 2 — Remove duplicate rows
before = df.shape[0]
df.drop_duplicates(inplace=True)
print(f'Duplicate rows removed: {before - df.shape[0]}')
print(f'Rows remaining: {df.shape[0]}')

In [ ]:
# Step 3 — Handle missing winner values
print(f'Missing winners before: {df["winner"].isnull().sum()}')

# Fill with 'Unknown' — these are historical gaps
df['winner'] = df['winner'].fillna('Unknown')

print(f'Missing winners after: {df["winner"].isnull().sum()}')

In [ ]:
# Step 4 — Fix data types
df['year'] = df['year'].astype(int)
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['is_person'] = df['is_person'].astype(bool)
df['is_album_or_single'] = df['is_album_or_single'].astype(bool)

print('Data types after fix:')
print(df.dtypes)

In [ ]:
# Step 5 — Clean text columns
df['winner'] = df['winner'].str.strip()
df['details'] = df['details'].str.strip()
df['location'] = df['location'].str.strip()
df['host'] = df['host'].str.strip()

print('Text columns cleaned.')

In [ ]:
# Step 6 — Normalize messy category names into broad groups
def categorize_award(detail):
    d = detail.lower()
    if 'album' in d or 'mastercard' in d:
        return 'Album'
    elif 'single' in d or 'song' in d or 'video' in d:
        return 'Single/Song/Video'
    elif 'female' in d or 'woman' in d:
        return 'Female Artist'
    elif 'male' in d or 'man' in d:
        return 'Male Artist'
    elif 'group' in d or 'band' in d or 'act' in d:
        return 'Group/Act'
    elif 'newcomer' in d or 'new artist' in d or 'breakthrough' in d or 'rising' in d or 'critic' in d:
        return 'Newcomer/Breakthrough'
    elif 'international' in d:
        return 'International'
    elif 'contribution' in d or 'achievement' in d or 'icon' in d or 'global' in d or 'special' in d or 'outstanding' in d or 'life' in d:
        return 'Lifetime/Special'
    elif 'producer' in d:
        return 'Producer'
    elif 'classical' in d or 'soundtrack' in d or 'comedy' in d:
        return 'Specialist'
    else:
        return 'Other'

df['category_group'] = df['details'].apply(categorize_award)
print('Category groups:')
print(df['category_group'].value_counts())

In [ ]:
# Step 7 — Add decade column
df['decade'] = (df['year'] // 10) * 10
print('Decade distribution:')
print(df['decade'].value_counts().sort_index())

In [ ]:
# Step 8 — Sort by year and reset index
df = df.sort_values('year').reset_index(drop=True)
print('Data sorted by year. Final shape:', df.shape)
df.head()

**Cleaning Summary:**
- Column names standardized to lowercase
- 49 missing winners filled with 'Unknown'
- Date column parsed to datetime format
- 100+ messy category names normalized into 10 broad `category_group` labels
- New `decade` feature engineered for temporal analysis

---

## Section 4 — Data Visualizations
Visual exploration of the dataset through charts and plots.

In [ ]:
# Chart 1 — Top 10 Most Awarded Artists
top_artists = df[df['is_person'] == True]['winner'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_artists.index[::-1], top_artists.values[::-1],
               color=sns.color_palette('viridis', 10))

for bar, val in zip(bars, top_artists.values[::-1]):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=11, fontweight='bold')

ax.set_title('Top 10 Most Awarded Artists at the BRITs (1982–2025)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Number of Awards')
ax.set_xlim(0, top_artists.max() + 1.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 2 — Number of Awards Given Per Year
awards_per_year = df.groupby('year').size()

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(awards_per_year.index, awards_per_year.values, alpha=0.3, color='steelblue')
ax.plot(awards_per_year.index, awards_per_year.values, color='steelblue', linewidth=2, marker='o', markersize=4)

# Annotate pandemic dip
ax.annotate('COVID-19\npandemic', xy=(2020, awards_per_year[2020]),
            xytext=(2017, 5), fontsize=9,
            arrowprops=dict(arrowstyle='->', color='red'),
            color='red')

ax.set_title('Number of Awards Given Per Year (1982–2025)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Year')
ax.set_ylabel('Number of Awards')
ax.xaxis.set_major_locator(mticker.MultipleLocator(5))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 3 — Award Category Group Distribution (Pie Chart)
cat_counts = df['category_group'].value_counts()

fig, ax = plt.subplots(figsize=(9, 7))
wedges, texts, autotexts = ax.pie(
    cat_counts.values,
    labels=cat_counts.index,
    autopct='%1.1f%%',
    startangle=140,
    colors=sns.color_palette('Set2', len(cat_counts)),
    pctdistance=0.82
)
for t in autotexts:
    t.set_fontsize(9)

ax.set_title('Distribution of Award Categories at the BRITs', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 4 — Top 8 Venues
venue_counts = df['location'].value_counts().head(8)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=venue_counts.values, y=venue_counts.index, palette='coolwarm', ax=ax)

for i, v in enumerate(venue_counts.values):
    ax.text(v + 1, i, str(v), va='center', fontsize=10)

ax.set_title('Top 8 BRIT Awards Venues (by Awards Given)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Total Awards Presented')
ax.set_ylabel('')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 5 — Top 8 Hosts
host_counts = df['host'].value_counts().head(8)

fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette('magma', len(host_counts))
bars = ax.barh(host_counts.index[::-1], host_counts.values[::-1], color=colors)

for bar, val in zip(bars, host_counts.values[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=10)

ax.set_title('Most Frequent BRIT Awards Hosts', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Years Hosted')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

**Visualizations Summary:**
- **U2** leads all-time with 7 wins followed by Annie Lennox (6) and several artists tied at 5
- Awards per year grew from ~10 in the 1980s to 12–17 in recent years, with a drop during COVID-19 in 2020
- **Group/Act** awards dominate the pie at ~25%, followed by International and Album categories
- **The O2** hosts the most modern BRITs; historic ceremonies were at Earls Court
- **Chris Evans** has hosted the most BRIT ceremonies, followed by Jack Whitehall and James Corden

---

## Section 5 — Trend Analysis
How have award categories, genders, and artist types evolved across decades?

In [ ]:
# Chart 6 — Category Group Trends by Decade (Stacked Bar)
decade_cat = df.groupby(['decade', 'category_group']).size().unstack(fill_value=0)

# Normalize to percentage
decade_cat_pct = decade_cat.div(decade_cat.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(13, 6))
decade_cat_pct.plot(kind='bar', stacked=True, ax=ax,
                    colormap='tab10', edgecolor='white', linewidth=0.5)

ax.set_title('Award Category Mix by Decade (% of total awards)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Decade')
ax.set_ylabel('Percentage (%)')
ax.set_xticklabels([f'{int(d)}s' for d in decade_cat_pct.index], rotation=0)
ax.legend(loc='upper left', bbox_to_anchor=(1, 1), title='Category Group')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 7 — Person vs Album/Single Awards Over Time
type_year = df.groupby(['year', 'is_person']).size().unstack(fill_value=0)
type_year.columns = ['Not a Person', 'Person']

fig, ax = plt.subplots(figsize=(14, 5))
type_year.plot(ax=ax, linewidth=2, marker='o', markersize=3)

ax.set_title('Awards to Persons vs Non-Persons (Albums/Acts) Per Year', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Year')
ax.set_ylabel('Number of Awards')
ax.legend(title='Winner Type')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 8 — Female vs Male Artist Awards per Decade
female = df[df['category_group'] == 'Female Artist'].groupby('decade').size()
male   = df[df['category_group'] == 'Male Artist'].groupby('decade').size()

gender_df = pd.DataFrame({'Female Artist': female, 'Male Artist': male}).fillna(0)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(gender_df))
width = 0.35

b1 = ax.bar(x - width/2, gender_df['Female Artist'], width, label='Female Artist', color='#e05a8a')
b2 = ax.bar(x + width/2, gender_df['Male Artist'], width, label='Male Artist', color='#4a90d9')

ax.set_title('Female vs Male Artist Awards per Decade', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Decade')
ax.set_ylabel('Number of Awards')
ax.set_xticks(x)
ax.set_xticklabels([f'{int(d)}s' for d in gender_df.index])
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 9 — International vs British awards trend over time (rolling 5-year)
intl = df[df['category_group'] == 'International'].groupby('year').size()
brit = df[df['category_group'] != 'International'].groupby('year').size()

intl_roll = intl.rolling(5, center=True).mean()
brit_roll = brit.rolling(5, center=True).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(intl_roll.index, intl_roll.values, label='International Awards (5yr avg)', color='orange', linewidth=2)
ax.plot(brit_roll.index, brit_roll.values, label='British/Other Awards (5yr avg)', color='royalblue', linewidth=2)

ax.set_title('International vs British Award Volume Over Time (5-Year Rolling Average)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Year')
ax.set_ylabel('Number of Awards (Avg)')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

**Trend Analysis Summary:**
- **Group/Act** categories have been the most consistent across all decades
- **International** categories peaked in the 1990s–2000s and have since declined
- Female and Male solo artist awards have been roughly balanced since the 1980s
- The 2020s show a shift towards broader, more inclusive categories (e.g., gender-neutral solo artist awards)

---

## Section 6 — Correlation & Pattern Analysis
Exploring relationships between features using heatmaps, frequency matrices and crosstabs.

In [ ]:
# Encode boolean columns as integers for correlation
corr_df = df[['year', 'is_person', 'is_album_or_single']].copy()
corr_df['is_person'] = corr_df['is_person'].astype(int)
corr_df['is_album_or_single'] = corr_df['is_album_or_single'].astype(int)

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
            linewidths=0.5, annot_kws={'size': 12})
ax.set_title('Correlation Matrix of Numeric Features', fontsize=13, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

In [ ]:
# Crosstab — Category Group vs Decade (heatmap)
ct = pd.crosstab(df['category_group'], df['decade'])
ct.columns = [f"{int(c)}s" for c in ct.columns]

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(ct, annot=True, fmt='d', cmap='YlOrRd', ax=ax,
            linewidths=0.5, linecolor='lightgrey')
ax.set_title('Award Category Group Frequency by Decade', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Decade')
ax.set_ylabel('Category Group')
plt.tight_layout()
plt.show()

In [ ]:
# Top artist win streaks — count wins and years active
artist_stats = df[df['winner'] != 'Unknown'].groupby('winner').agg(
    total_wins=('winner', 'count'),
    first_win=('year', 'min'),
    last_win=('year', 'max')
).query('total_wins >= 3').sort_values('total_wins', ascending=False)

artist_stats['career_span'] = artist_stats['last_win'] - artist_stats['first_win']
print('Artists with 3+ BRIT Awards:')
print(artist_stats.head(15).to_string())

In [ ]:
# Chart 10 — Win Span vs Total Wins for top artists
plot_df = artist_stats.head(20)

fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(plot_df['career_span'], plot_df['total_wins'],
                     s=plot_df['total_wins'] * 60, alpha=0.7,
                     c=plot_df['total_wins'], cmap='plasma')

for _, row in plot_df.iterrows():
    ax.annotate(row.name, (row['career_span'], row['total_wins']),
                fontsize=8, ha='left', va='bottom',
                xytext=(3, 3), textcoords='offset points')

ax.set_title('Career Span vs Total BRIT Awards (Top Artists)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Career Span at BRITs (years)')
ax.set_ylabel('Total Awards Won')
plt.colorbar(scatter, ax=ax, label='Total Awards')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

**Correlation Analysis Summary:**
- `is_person` and `is_album_or_single` are slightly negatively correlated — some awards are neither
- The heatmap shows **International** awards concentrated in the 1990s and 2000s, declining in the 2020s
- **Newcomer/Breakthrough** awards appear consistently across all decades
- Long-running artists like U2 and Annie Lennox have career spans exceeding 20 years at the BRITs

---

## Section 7 — Machine Learning: Award Type Classification
We use features available at the time of the award to classify whether the winner is a **person** (`is_person = True`) or not (album, group, etc.). We compare **Logistic Regression** and **Random Forest**.

In [ ]:
# Feature Engineering for ML
ml_df = df.copy()

# Encode categorical features
le_cat = LabelEncoder()
le_host = LabelEncoder()
le_venue = LabelEncoder()

ml_df['category_enc'] = le_cat.fit_transform(ml_df['category_group'])
ml_df['host_enc'] = le_host.fit_transform(ml_df['host'])
ml_df['venue_enc'] = le_venue.fit_transform(ml_df['location'])

# Features and target
features = ['year', 'decade', 'category_enc', 'host_enc', 'venue_enc', 'is_album_or_single']
target = 'is_person'

X = ml_df[features].astype(float)
y = ml_df[target].astype(int)

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training samples : {X_train.shape[0]}')
print(f'Testing  samples : {X_test.shape[0]}')
print(f'Features used    : {features}')

In [ ]:
# Model 1 — Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

lr_acc = accuracy_score(y_test, y_pred_lr)
print(f'Logistic Regression Accuracy: {lr_acc:.4f} ({lr_acc*100:.2f}%)')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred_lr, target_names=['Not a Person', 'Person']))

In [ ]:
# Model 2 — Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

rf_acc = accuracy_score(y_test, y_pred_rf)
print(f'Random Forest Accuracy: {rf_acc:.4f} ({rf_acc*100:.2f}%)')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred_rf, target_names=['Not a Person', 'Person']))

In [ ]:
# Confusion Matrix Comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_pred, title in zip(axes,
                              [y_pred_lr, y_pred_rf],
                              ['Logistic Regression', 'Random Forest']):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Not a Person', 'Person'],
                yticklabels=['Not a Person', 'Person'],
                linewidths=0.5)
    ax.set_title(f'{title}\nAccuracy: {accuracy_score(y_test, y_pred)*100:.2f}%',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrix — Award Type Classification', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance — Random Forest
feature_imp = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
feature_imp.plot(kind='barh', ax=ax, color=sns.color_palette('viridis', len(feature_imp)))

ax.set_title('Feature Importance — Random Forest Classifier', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Importance Score')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Model Accuracy Comparison Bar Chart
model_names = ['Logistic Regression', 'Random Forest']
accuracies  = [lr_acc * 100, rf_acc * 100]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(model_names, accuracies, color=['#4a90d9', '#e07a3a'], width=0.4, edgecolor='white')

for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 3,
            f'{acc:.2f}%', ha='center', va='top', fontsize=13, fontweight='bold', color='white')

ax.set_ylim(0, 110)
ax.set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Accuracy (%)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

**ML Summary:**
- Both models classify award type (person vs non-person) well
- **Random Forest** outperforms Logistic Regression by leveraging non-linear feature interactions
- The most important feature is `is_album_or_single` — which directly encodes award type, acting as a strong signal
- `category_enc` (the award category group) is the second most informative feature
- `year` and `decade` add temporal context to the predictions

---

## Section 8 — Conclusion

### Key Findings

| Insight | Finding |
|---|---|
| 🏆 Most Decorated Artist | **U2** with 7 BRIT Awards |
| 📅 Year Range | 1982 – 2025 (43 years) |
| 🎭 Total Categories (raw) | 100+ evolving names → 10 groups |
| 📍 Most Common Venue | The O2, London |
| 🎤 Most Frequent Host | Chris Evans |
| 📉 Smallest Ceremony | 2020 (COVID-19, 8 awards) |
| 🌍 International Category Trend | Peaked 1990s–2000s, declining since |
| 🤖 Best ML Model | Random Forest |

### What We Built
- ✅ Cleaned and standardized 43 years of messy award data
- ✅ Engineered `category_group` and `decade` features
- ✅ Created 10 visualizations covering winners, trends, venues, hosts and gender
- ✅ Built and compared two ML classifiers to predict award type

### Future Work
- Sentiment analysis of award category descriptions over time
- Predicting future BRIT Award winners using external music chart data
- Network analysis of artists who won multiple categories in the same year